In [6]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.lightgbm
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error

# Set MLflow experiment — Fabric auto-creates this in the workspace
mlflow.set_experiment("retail-demand-forecasting-s01")

print("MLflow tracking URI:", mlflow.get_tracking_uri())

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 11, Finished, Available, Finished, False)

2026/04/24 10:15:29 INFO mlflow.tracking.fluent: Experiment with name 'retail-demand-forecasting-s01' does not exist. Creating a new experiment.


MLflow tracking URI: sds://pbipindcen4-centralindia.pbidedicated.windows.net/webapi/capacities/5e50d93c-9a81-4c90-b011-59017af54c5f/workloads/ML/ML/Automatic/workspaceid/4cca3437-11dd-478f-b56b-5fb278b8c38a/


##  Load the feature table from the Lakehouse

In [7]:
# Load from Lakehouse as Pandas
df = spark.table("fact_store_item_features").toPandas()

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Date range:", df['date'].min(), "→", df['date'].max())
print("Nulls:\n", df.isnull().sum())

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 12, Finished, Available, Finished, False)

Shape: (3054348, 22)
Columns: ['date', 'store_nbr', 'item_id', 'family', 'sales', 'onpromotion', 'is_holiday', 'holiday_type', 'year', 'month', 'day_of_week', 'city', 'state', 'store_type', 'cluster', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'ma_7', 'ma_14', 'ma_28']
Date range: 2013-01-01 → 2017-08-15
Nulls:
 date                  0
store_nbr             0
item_id               0
family                0
sales                 0
onpromotion           0
is_holiday            0
holiday_type    2551824
year                  0
month                 0
day_of_week           0
city                  0
state                 0
store_type            0
cluster               0
lag_1              1782
lag_7             12474
lag_14            24948
lag_28            49896
ma_7                  0
ma_14                 0
ma_28                 0
dtype: int64


In [8]:
display(df)

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 2166d937-bdb1-49cf-96b6-52adcbc614ba)

## preprocessing

In [9]:
# Parse date
df['date'] = pd.to_datetime(df['date'])

# Encode categoricals
df['day_of_week'] = df['date'].dt.dayofweek  # 0=Mon, 6=Sun
df['is_holiday'] = df['is_holiday'].astype(int)
df['onpromotion'] = df['onpromotion'].fillna(0).astype(int)

# Label-encode family, city, state, store_type
cat_cols = ['family', 'city', 'state', 'store_type']
for col in cat_cols:
    df[col] = df[col].astype('category').cat.codes

# Fill nulls in lags/MAs with -1 (missing signal marker for tree model)
lag_ma_cols = ['lag_1', 'lag_7', 'lag_14', 'lag_28', 'ma_7', 'ma_14', 'ma_28']
df[lag_ma_cols] = df[lag_ma_cols].fillna(-1)

# Define features and target
FEATURES = [
    'store_nbr', 'item_id', 'family', 'onpromotion',
    'is_holiday', 'year', 'month', 'day_of_week',
    'city', 'state', 'store_type', 'cluster',
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'ma_7', 'ma_14', 'ma_28'
]
TARGET = 'sales'

print("Features ready:", FEATURES)

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 14, Finished, Available, Finished, False)

Features ready: ['store_nbr', 'item_id', 'family', 'onpromotion', 'is_holiday', 'year', 'month', 'day_of_week', 'city', 'state', 'store_type', 'cluster', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'ma_7', 'ma_14', 'ma_28']


## time‑based split

In [10]:
# Use last 28 days as validation; rest as training
# This is the correct approach for time-series — no random split
VALIDATION_CUTOFF = df['date'].max() - pd.Timedelta(days=28)

train = df[df['date'] <= VALIDATION_CUTOFF].copy()
val   = df[df['date'] >  VALIDATION_CUTOFF].copy()

X_train, y_train = train[FEATURES], train[TARGET]
X_val,   y_val   = val[FEATURES],   val[TARGET]

print(f"Training rows : {len(train)}")
print(f"Validation rows: {len(val)}")
print(f"Train date range: {train['date'].min()} → {train['date'].max()}")
print(f"Val   date range: {val['date'].min()} → {val['date'].max()}")

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 15, Finished, Available, Finished, False)

Training rows : 3004452
Validation rows: 49896
Train date range: 2013-01-01 00:00:00 → 2017-07-18 00:00:00
Val   date range: 2017-07-19 00:00:00 → 2017-08-15 00:00:00


###### LightGBM (Light Gradient Boosting Machine) is an open-source, high-performance machine learning framework developed by Microsoft. It is based on decision tree algorithms and is used for supervised learning tasks such as classification, regression, and ranking. The "Light" in its name refers to its primary focus on high speed and low memory consumption.

## Train LightGBM model with MLflow tracking

In [11]:
params = {
    "objective": "regression",
    "metric": "rmse",
    "learning_rate": 0.05,
    "num_leaves": 64,
    "n_estimators": 500,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
    "verbose": -1
}

with mlflow.start_run(run_name="lgbm_baseline_run_01") as run:
    RUN_ID = run.info.run_id
    print("MLflow Run ID:", RUN_ID)

    # Log hyperparameters
    mlflow.log_params(params)

    # Train model
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
    )

    # Predict on validation set
    val_preds = model.predict(X_val)

    # Evaluate
    rmse = np.sqrt(mean_squared_error(y_val, val_preds))
    mape = mean_absolute_percentage_error(y_val[y_val > 0], val_preds[y_val > 0])  # avoid div/0

    mlflow.log_metric("val_rmse", round(rmse, 4))
    mlflow.log_metric("val_mape", round(mape, 4))

    # Log and register model
    mlflow.lightgbm.log_model(
        model,
        artifact_path="retail_lgbm_model",
        registered_model_name="retail-demand-lgbm-s01"
    )

    print(f"Val RMSE: {rmse:.4f}")
    print(f"Val MAPE: {mape:.4f}")

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 16, Finished, Available, Finished, False)

MLflow Run ID: 1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b
Training until validation scores don't improve for 50 rounds
[100]	valid_0's rmse: 219.237
[200]	valid_0's rmse: 212.342
[300]	valid_0's rmse: 211.141
[400]	valid_0's rmse: 209.632
Early stopping, best iteration is:
[389]	valid_0's rmse: 208.89
Val RMSE: 208.8901
Val MAPE: 0.3540


/home/trusted-service-user/cluster-env/trident_env/lib/python3.11/site-packages/_distutils_hack/__init__.py:33: UserWarning: Setuptools is replacing distutils.
  warnings.warn("Setuptools is replacing distutils.")
Successfully registered model 'retail-demand-lgbm-s01'.


## 

## Score validation and write forecast table to Lakehouse

In [12]:
from pyspark.sql import functions as F

# Prepare forecast DataFrame
forecast_df = val[['date', 'store_nbr', 'item_id', 'family']].copy()
forecast_df['forecast_qty']  = np.maximum(val_preds, 0)   # clip negatives
forecast_df['actual_sales']  = y_val.values
forecast_df['model_run_id']  = RUN_ID
forecast_df['model_name']    = "lgbm_baseline"
forecast_df['created_at']    = pd.Timestamp.now()

print(forecast_df.head())
print("Negative forecasts clipped:", (val_preds < 0).sum())

# Write to Lakehouse as Delta table
forecast_spark = spark.createDataFrame(forecast_df)
forecast_spark.write.mode("overwrite").format("delta").saveAsTable("fact_store_item_forecast")

print("fact_store_item_forecast written to Lakehouse")

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 18, Finished, Available, Finished, False)

           date  store_nbr  item_id  family  forecast_qty  actual_sales  \
1686 2017-07-19          1       19      15     18.681967          16.0   
1687 2017-07-20          1       19      15     22.456543          25.0   
1688 2017-07-21          1       19      15     30.313693          47.0   
1689 2017-07-22          1       19      15     24.728103          15.0   
1690 2017-07-23          1       19      15     16.608526           6.0   

                              model_run_id     model_name  \
1686  1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b  lgbm_baseline   
1687  1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b  lgbm_baseline   
1688  1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b  lgbm_baseline   
1689  1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b  lgbm_baseline   
1690  1ccfbbb9-29ac-4e25-8b94-d63c4f353a1b  lgbm_baseline   

                     created_at  
1686 2026-04-24 10:23:42.430373  
1687 2026-04-24 10:23:42.430373  
1688 2026-04-24 10:23:42.430373  
1689 2026-04-24 10:23:42.430373  
1690 2026-04

In [14]:
display(forecast_spark)

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 21, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 05e88c1d-5ffb-4407-b17c-688206d5c096)

In [15]:
import pandas as pd

importance = pd.DataFrame({
    "feature": FEATURES,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

print(importance.to_string(index=False))

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 22, Finished, Available, Finished, False)

    feature  importance
      lag_1        2551
onpromotion        2478
       ma_7        2212
day_of_week        2210
      month        2122
      lag_7        1903
     lag_28        1741
  store_nbr        1638
     lag_14        1440
      ma_14        1036
      ma_28         844
       year         834
 is_holiday         757
    cluster         727
    item_id         530
     family         468
 store_type         355
       city         335
      state         326


## Load and verify forecast table

In [16]:
spark.table("fact_store_item_forecast").show(10)
spark.table("fact_store_item_forecast").printSchema()

StatementMeta(, aab67f90-0151-456f-882d-bb7a49a22a1f, 23, Finished, Available, Finished, False)

+-------------------+---------+-------+------+------------------+------------+--------------------+-------------+--------------------+
|               date|store_nbr|item_id|family|      forecast_qty|actual_sales|        model_run_id|   model_name|          created_at|
+-------------------+---------+-------+------+------------------+------------+--------------------+-------------+--------------------+
|2017-07-26 00:00:00|        8|     25|    25|279.64625169217715|       215.0|1ccfbbb9-29ac-4e2...|lgbm_baseline|2026-04-24 10:23:...|
|2017-07-27 00:00:00|        8|     25|    25|248.62070533175535|       236.0|1ccfbbb9-29ac-4e2...|lgbm_baseline|2026-04-24 10:23:...|
|2017-07-28 00:00:00|        8|     25|    25| 319.8135161853528|       300.0|1ccfbbb9-29ac-4e2...|lgbm_baseline|2026-04-24 10:23:...|
|2017-07-29 00:00:00|        8|     25|    25|407.68172539319823|       410.0|1ccfbbb9-29ac-4e2...|lgbm_baseline|2026-04-24 10:23:...|
|2017-07-30 00:00:00|        8|     25|    25| 438.3502

In [ ]:
display(fact)